# TrafficTwin E1 Colab GPU backend smoke

This notebook is limited to two identical 10-step, 2.5x, fleet-seed-0 physical validation runs. It must not launch a 3,600-step run or campaign cell. Before running, choose **Runtime → Change runtime type → NVIDIA GPU**. The uploaded archive is private research input: do not share or save it to a public Drive.


In [ ]:
import shutil
import subprocess

nvidia_smi = shutil.which("nvidia-smi")
assert nvidia_smi, "Stop: this is not a Colab NVIDIA GPU runtime"
subprocess.run([nvidia_smi], check=True)  # noqa: S603

Install the exact scientific packages. If Colab says packages already imported must be restarted, use **Runtime → Restart session**, then continue from the device/version cell; do not rerun a completed smoke.


In [ ]:
%pip install --quiet --upgrade \
  "jax[cuda12]==0.4.30" "jaxlib==0.4.30" \
  "jax-cuda12-plugin[with-cuda]==0.4.30" "jax-cuda12-pjrt==0.4.30" \
  "numpy==1.26.4" "scipy==1.17.1"

In [ ]:
import importlib.metadata
import platform

import jax

required = {
    "jax": "0.4.30",
    "jaxlib": "0.4.30",
    "jax-cuda12-plugin": "0.4.30",
    "jax-cuda12-pjrt": "0.4.30",
    "numpy": "1.26.4",
    "scipy": "1.17.1",
}
observed = {name: importlib.metadata.version(name) for name in required}
assert observed == required, (observed, required)
assert jax.default_backend() == "gpu", (jax.default_backend(), jax.devices())
print(
    {
        "python": platform.python_version(),
        "packages": observed,
        "devices": [str(device) for device in jax.devices()],
        "device_kinds": [device.device_kind for device in jax.devices()],
    }
)

Upload exactly `e1-colab-gpu-backend-smoke-v1-inputs-r3.tar.gz` from the local private bundle. The cell refuses any different byte identity.


In [ ]:
import hashlib
import shutil
import subprocess
from pathlib import Path

from google.colab import files

expected_name = "e1-colab-gpu-backend-smoke-v1-inputs-r3.tar.gz"
expected_sha256 = "7cf78ed99f912bfbcd7d50f08a1dd2baa0c89a9de8260a5bf05be9d3838b6091"
uploaded = files.upload()
assert set(uploaded) == {expected_name}, uploaded.keys()
payload = uploaded[expected_name]
assert hashlib.sha256(payload).hexdigest() == expected_sha256
archive = Path("/content") / expected_name
archive.write_bytes(payload)
tar_binary = shutil.which("tar")
assert tar_binary
subprocess.run(  # noqa: S603
    [tar_binary, "-xzf", str(archive), "-C", "/content"],
    check=True,
)

In [ ]:
import subprocess
import sys

bundle = "/content/e1_colab_bundle"
smoke_argv = [
    sys.executable,
    f"{bundle}/traffictwin/scripts/run_e1_colab_gpu_backend_smoke.py",
    "--manifest",
    f"{bundle}/traffictwin/docs/evaluation/e1/e1_colab_gpu_backend_smoke_manifest_v1.json",
    "--bundle-root",
    bundle,
    "--output-root",
    "/content/e1-colab-gpu-backend-smoke-v1",
]
result = subprocess.run(smoke_argv, check=False)  # noqa: S603
print({"smoke_exit_code": result.returncode})

In [ ]:
import json

report_path = Path("/content/e1-colab-gpu-backend-smoke-v1/colab_backend_smoke_report.json")
if report_path.exists():
    report = json.loads(report_path.read_text())
    summary = {
        "passed": report["passed"],
        "backend": report["backend"],
        "smoke_started": report.get("smoke_started", True),
        "input_identity": report["input_identity"],
        "package_identity": report["package_identity"],
        "speed": report.get("speed"),
        "recommendation": report["recommendation"],
        "cross_backend_comparison": report.get("cross_backend_comparison"),
    }
    print(json.dumps(summary, indent=2))
else:
    print("Backend report unavailable; download the retained failure directory.")

In [ ]:
import shutil

from google.colab import files

result_archive = shutil.make_archive(
    "/content/e1-colab-gpu-backend-smoke-v1-results",
    "zip",
    "/content/e1-colab-gpu-backend-smoke-v1",
)
files.download(result_archive)